# AlphaFold2 Structure Prediction: Kunitz Domain

This notebook runs ColabFold (AlphaFold2-PTM) on Kunitz domain sequences:
- **Stored** (50 natural sequences from PF00014)
- **SA strong-conditioned** (50 SA-generated, conditioned on trypsin binders)
- **SA full-family** (50 SA-generated, full-family seeding)

Reference structure: 1BPI chain A (BPTI, crystal structure)

**Requirements:** Run on Google Colab with a **T4 GPU** runtime.

---

## 1. Install ColabFold

In [ ]:
%%bash
# Remove TF to avoid JAX/TF kernel conflicts in Colab
pip uninstall -q -y tensorflow tensorflow-gpu tf-keras 2>/dev/null || true

%%bash
# Install ColabFold (takes ~3-5 minutes)
if ! command -v colabfold_batch &> /dev/null; then
    pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
    pip install -q "jax[cuda12]"
fi
colabfold_batch --help | head -3

## 2. Upload Input FASTA Files

Upload the three FASTA files from `code/data/af2_input/kunitz/`:
- `stored.fasta`
- `sa_strong.fasta`
- `sa_full.fasta`

In [ ]:
import os
from google.colab import files

# Create working directories
WORK = "/content/kunitz_af2"
os.makedirs(f"{WORK}/input", exist_ok=True)
os.makedirs(f"{WORK}/output", exist_ok=True)
os.makedirs(f"{WORK}/results", exist_ok=True)

# Upload FASTA files
print("Upload the 3 FASTA files (stored.fasta, sa_strong.fasta, sa_full.fasta):")
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f"{WORK}/input/{fname}", 'wb') as f:
        f.write(content)
    n_seqs = content.decode().count('>')
    print(f"  {fname}: {n_seqs} sequences")

## 3. Download Reference Structure (1BPI)

In [ ]:
import urllib.request

# Download 1BPI (BPTI crystal structure)
REF_PDB = f"{WORK}/1BPI.pdb"
urllib.request.urlretrieve(
    "https://files.rcsb.org/download/1BPI.pdb",
    REF_PDB
)

# Extract chain A only
REF_PDB_A = f"{WORK}/1BPI_A.pdb"
with open(REF_PDB) as fin, open(REF_PDB_A, 'w') as fout:
    for line in fin:
        if (line.startswith('ATOM') or line.startswith('TER')) and len(line) > 21 and line[21] == 'A':
            fout.write(line)
        elif line.startswith('END'):
            fout.write(line)

print(f"Reference: {REF_PDB_A}")

## 4. Install TM-align

In [ ]:
%%bash
if ! command -v TMalign &> /dev/null; then
    wget -q https://zhanggroup.org/TM-align/TMalign.cpp -O /tmp/TMalign.cpp
    g++ -O3 -o /usr/local/bin/TMalign /tmp/TMalign.cpp
fi
TMalign -h 2>&1 | head -3

## 5. Run ColabFold AF2-PTM

This runs AlphaFold2-PTM on all 150 sequences (~30-45 min on T4).

In [ ]:
import subprocess
import time

CATEGORIES = ["stored", "sa_strong", "sa_full"]

for cat in CATEGORIES:
    fasta = f"{WORK}/input/{cat}.fasta"
    outdir = f"{WORK}/output/{cat}"
    os.makedirs(outdir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Running ColabFold on: {cat}")
    print(f"{'='*60}")

    t0 = time.time()
    result = subprocess.run([
        "colabfold_batch",
        fasta,
        outdir,
        "--model-type", "alphafold2_ptm",
        "--num-models", "1",
        "--num-recycle", "3",
        "--model-order", "1",
        "--num-relax", "0",
        "--msa-mode", "mmseqs2_uniref_env",
    ], capture_output=True, text=True)

    elapsed = time.time() - t0
    n_pdbs = len([f for f in os.listdir(outdir) if f.endswith('.pdb')])
    print(f"  Done: {n_pdbs} PDBs in {elapsed:.0f}s")

    if result.returncode != 0:
        print(f"  WARNING: colabfold_batch returned {result.returncode}")
        print(result.stderr[-500:] if result.stderr else "no stderr")

## 6. Extract pLDDT and Compute TM-scores

In [ ]:
import re
import csv
import numpy as np

def extract_mean_plddt(pdb_path):
    """Extract mean pLDDT from B-factor column of ATOM records."""
    plddts = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith('ATOM') and len(line) >= 66:
                try:
                    plddt = float(line[60:66].strip())
                    plddts.append(plddt)
                except ValueError:
                    pass
    if not plddts:
        return None
    m = np.mean(plddts)
    return m * 100.0 if m < 1.5 else m


def run_tmalign(query_pdb, ref_pdb):
    """Run TM-align and return TM-score normalized by reference length."""
    result = subprocess.run(
        ["TMalign", query_pdb, ref_pdb],
        capture_output=True, text=True
    )
    for line in result.stdout.split('\n'):
        if 'TM-score=' in line and 'Chain_2' in line:
            m = re.search(r'TM-score=\s*([\d.]+)', line)
            if m:
                return float(m.group(1))
    return None


# Collect results
results = []
for cat in CATEGORIES:
    outdir = f"{WORK}/output/{cat}"
    pdbs = sorted([f for f in os.listdir(outdir) if f.endswith('.pdb') and 'unrelaxed' in f])

    if not pdbs:
        pdbs = sorted([f for f in os.listdir(outdir) if f.endswith('.pdb')])

    print(f"\n{cat}: {len(pdbs)} PDB files")
    for pdb_file in pdbs:
        pdb_path = os.path.join(outdir, pdb_file)
        plddt = extract_mean_plddt(pdb_path)
        tmscore = run_tmalign(pdb_path, REF_PDB_A)

        name = pdb_file.replace('_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb', '')
        name = name.replace('.pdb', '')

        results.append({
            'name': name,
            'source': cat,
            'plddt': plddt,
            'tmscore': tmscore,
            'predictor': 'AF2'
        })
        print(f"  {name}: pLDDT={plddt:.1f}, TM={tmscore:.4f}")

# Save raw results
csv_path = f"{WORK}/results/kunitz_af2_validation_raw.csv"
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'source', 'plddt', 'tmscore', 'predictor'])
    writer.writeheader()
    writer.writerows(results)
print(f"\nRaw results saved to: {csv_path}")

## 7. Summary Statistics

In [ ]:
print(f"{'Source':<15} {'n':>3}  {'pLDDT':>15}  {'TM-score':>15}")
print("-" * 55)
for cat in CATEGORIES:
    subset = [r for r in results if r['source'] == cat]
    plddts = [r['plddt'] for r in subset if r['plddt'] is not None]
    tms = [r['tmscore'] for r in subset if r['tmscore'] is not None]
    print(f"{cat:<15} {len(subset):>3}  "
          f"{np.mean(plddts):>6.1f} ± {np.std(plddts):>4.1f}  "
          f"{np.mean(tms):>6.4f} ± {np.std(tms):.4f}")

# Save summary
summary_path = f"{WORK}/results/kunitz_af2_validation_summary.csv"
with open(summary_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['source', 'n', 'plddt_mean', 'plddt_std', 'tmscore_mean', 'tmscore_std'])
    for cat in CATEGORIES:
        subset = [r for r in results if r['source'] == cat]
        plddts = [r['plddt'] for r in subset if r['plddt'] is not None]
        tms = [r['tmscore'] for r in subset if r['tmscore'] is not None]
        writer.writerow([cat, len(subset), f"{np.mean(plddts):.1f}", f"{np.std(plddts):.1f}",
                        f"{np.mean(tms):.4f}", f"{np.std(tms):.4f}"])
print(f"\nSummary saved to: {summary_path}")

## 8. Download Results

In [ ]:
# Download CSV results
files.download(f"{WORK}/results/kunitz_af2_validation_raw.csv")
files.download(f"{WORK}/results/kunitz_af2_validation_summary.csv")

# Also zip all PDB files for download
import shutil
shutil.make_archive(f"{WORK}/results/kunitz_af2_pdbs", 'zip', f"{WORK}/output")
files.download(f"{WORK}/results/kunitz_af2_pdbs.zip")
print("Done! Download the CSV files and PDB zip.")